#### 1. File paths
The locations and filenames of the five CISA manuals are defined.

In [5]:
import re
import pandas as pd
from docx import Document
from pathlib import Path

data_dir = Path(r"C:\Users\liisiparve\Projects\cybersecurity-project\data\raw")

files = {
    "Healthcare": ("healthcare_and_public_health.docx", "HC"),
    "Vendor": ("vendor_supply_chain_compromise.docx", "VD"),
    "Ransomware": ("ransomware.docx", "RW"),
    "Insider Threat": ("insider_threat.docx", "IT"),
    "Executives": ("executives_and_senior_leadership.docx", "EX")
}

assert all((data_dir / filename).is_file() for filename, _ in files.values())

#### 2. Extracting the data
A function is used to identify scenario events and discussion questions in the CISA manuals.

In [6]:
def parse_manual(path, manual_name, prefix):
    doc = Document(path)

    # keep only the paragraphs that aren't empty
    paragraphs = [
        (i, p.text.strip())
        for i, p in enumerate(doc.paragraphs)
        if p.text.strip()
    ]

    events = []
    questions = []

    section = None # current module or vignette
    day = None # current scenario day
    mode = None # whether we're reading events or questions
    event_parts = [] # paragraphs belonging to the current event
    event_index = None # paragraph index where the event starts
    started = False # whether we've reached the actual exercise

    def save_event():
        if event_parts and section is not None:
            events.append({
                "manual_title": manual_name,
                "inject_id": f"{prefix}-INJ-{event_index}",
                "section": section,
                "day": day,
                "event_description": " ".join(event_parts),
                "source_paragraph": event_index
            })

    for idx, text in paragraphs:

        # find the beginning of the actual exercise
        if not started:
            if manual_name == "Executives":
                start_heading = "General Discussion Questions"
            else:
                start_heading = "Module 1"

            if text == start_heading:
                started = True
                section = text

                if manual_name == "Executives":
                    mode = "question"
                else:
                    mode = "event"

            continue

        # stop before the appendices
        if re.fullmatch(
            r"Appendix\s+A\s*:.*",
            text,
            re.I
        ):
            save_event()
            break

        # recognize modules and vignettes
        is_module = re.fullmatch(
            r"Module\s+[12](?:\s*[–-].*)?",
            text,
            re.I
        )

        is_vignette = re.fullmatch(
            r"Vignette\s+[12]\s*:.*",
            text,
            re.I
        )

        if is_module or is_vignette:
            save_event()

            section = text
            day = None
            mode = "event"
            event_parts = []
            event_index = None
            continue

        # recognize discussion question headings
        if text == "Discussion Questions":
            save_event()
            event_parts = []
            mode = "question"
            continue

        # recognize scenario days
        if re.fullmatch(r"Day\s+\d+.*", text, re.I):
            save_event()

            day = text
            event_index = idx
            event_parts = []
            mode = "event"
            continue

        # recognize scenario headings in Executives manual
        if text == "Scenario" and manual_name == "Executives":
            mode = "event"
            continue

        # skip introductory descriptions of questions
        if (
            text.lower().startswith(
                "discussion questions included"
            )
            or text.lower().startswith(
                "the general discussion questions below"
            )
        ):
            continue

        # recognize questions and discussion prompts
        is_prompt = (
            "?" in text
            or bool(re.match(
                r"^(Describe|Discuss|Explain|Identify|List|"
                r"Based on|Using your|What|How|Who|When|"
                r"Where|Does|Do|Are|Is|Have|Has|"
                r"Can|Could|Would|Should)\b",
                text,
                re.I
            ))
        )

        # if a question follows an event, switch modes
        if mode == "event" and is_prompt:
            save_event()
            event_parts = []
            mode = "question"

        # save discussion questions
        if mode == "question":
            if is_prompt:
                questions.append({
                    "manual_title": manual_name,
                    "question_id": f"{prefix}-Q-{idx}",
                    "section": section,
                    "day": day,
                    "question_text": text,
                    "domain_category": None,
                    "difficulty_level": None,
                    "required_skill": None,
                    "source_paragraph": idx
                })

        # save event descriptions
        elif mode == "event":
            if manual_name != "Executives" and day is None:
                continue

            if event_index is None:
                event_index = idx

            event_parts.append(text)

    return pd.DataFrame(events), pd.DataFrame(questions)

#### 3. Processing the manuals
The extraction function is applied to all five manuals, and the results are combined into two DataFrames.

In [7]:
all_events = []
all_questions = []

for name, (filename, prefix) in files.items():
    events, questions = parse_manual(data_dir / filename, name, prefix)

    all_events.append(events)
    all_questions.append(questions)

    print(f"{name}: {len(events)} events, {len(questions)} questions")

df_injects = pd.concat(all_events, ignore_index=True)
df_questions = pd.concat(all_questions, ignore_index=True)

print("\nTotal events:", len(df_injects))
print("Total questions:", len(df_questions))

display(df_injects.head())
display(df_questions.head())

Healthcare: 11 events, 61 questions
Vendor: 8 events, 49 questions
Ransomware: 8 events, 58 questions
Insider Threat: 12 events, 52 questions
Executives: 2 events, 48 questions

Total events: 41
Total questions: 268


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Healthcare,HC-INJ-87,Module 1,Day 1,The Cybersecurity and Infrastructure Security ...,87
1,Healthcare,HC-INJ-89,Module 1,Day 9,<Organization> employees receive e-mail notifi...,89
2,Healthcare,HC-INJ-117,Module 1,Day 10,Manufacturer’s updates are pushed for installa...,117
3,Healthcare,HC-INJ-119,Module 1,Day 13,A third-party electronic medical record (EMR) ...,119
4,Healthcare,HC-INJ-126,Module 1,Day 15,Radiology reports an issue with their Magnetic...,126


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Healthcare,HC-Q-93,Module 1,Day 9,What are the greatest cyber threats to your or...,None,None,None,93
1,Healthcare,HC-Q-94,Module 1,Day 9,What cybersecurity threat information does you...,None,None,None,94
2,Healthcare,HC-Q-95,Module 1,Day 9,What cyber threat information is most useful?,None,None,None,95
3,Healthcare,HC-Q-96,Module 1,Day 9,How is information disseminated throughout you...,None,None,None,96
4,Healthcare,HC-Q-97,Module 1,Day 9,What actions would your organization take in r...,None,None,None,97


In [8]:
for name, events, questions in zip(
    files.keys(),
    all_events,
    all_questions
):
    print(f"\n--- {name} ---")

    print("\nEvents:")
    display(events.head(3))

    print("\nDiscussion questions:")
    display(questions.head(3))


--- Healthcare ---

Events:


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Healthcare,HC-INJ-87,Module 1,Day 1,The Cybersecurity and Infrastructure Security ...,87
1,Healthcare,HC-INJ-89,Module 1,Day 9,<Organization> employees receive e-mail notifi...,89
2,Healthcare,HC-INJ-117,Module 1,Day 10,Manufacturer’s updates are pushed for installa...,117



Discussion questions:


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Healthcare,HC-Q-93,Module 1,Day 9,What are the greatest cyber threats to your or...,None,None,None,93
1,Healthcare,HC-Q-94,Module 1,Day 9,What cybersecurity threat information does you...,None,None,None,94
2,Healthcare,HC-Q-95,Module 1,Day 9,What cyber threat information is most useful?,None,None,None,95



--- Vendor ---

Events:


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Vendor,VD-INJ-111,Module 1,Day 1,CISA and the Federal Bureau of Investigation (...,111
1,Vendor,VD-INJ-122,Module 1,Day 7,Your organization contracts a new MSP who is k...,122
2,Vendor,VD-INJ-136,Module 1,Day 8,Data from multiple sector partners is found on...,136


... (10 outputs left out)


--- Executives ---

Events:


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Executives,EX-INJ-109,Vignette 1: Whaling/Ransomware,None,A senior finance employee receives what appear...,109
1,Executives,EX-INJ-126,Vignette 2: Insider Threat,None,Your IT department determines that a known exp...,126



Discussion questions:


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Executives,EX-Q-86,General Discussion Questions,None,What are the greatest cyber threats to your or...,None,None,None,86
1,Executives,EX-Q-87,General Discussion Questions,None,Describe your cybersecurity culture.,None,None,None,87
2,Executives,EX-Q-88,General Discussion Questions,None,How do you articulate the importance of a stro...,None,None,None,88


#### 4. Checking the extracted data
The extracted questions are compared with the original documents, and record IDs are checked for duplicates.

In [9]:
# Compare extracted questions with the original paragraphs
for _, row in df_questions.iterrows():
    filename = files[row["manual_title"]][0]
    doc = Document(data_dir / filename)

    original = doc.paragraphs[int(row["source_paragraph"])].text.strip()

    assert original == row["question_text"], f"Mismatch: {row['question_id']}"

# Check for duplicate IDs
assert df_injects["inject_id"].is_unique
assert df_questions["question_id"].is_unique

print("All validation checks passed!")

All validation checks passed!
